# 🚀 Building Useful AI Agents with Agno + OpenRouter
## Notebook 2 — Advanced (Stateful & Production-Ready)

In notebook 1 we built agents that **think and act**. But they were *stateless* — every
run started from a blank slate. This notebook makes agents **remember, learn, and persist**
(slides 11–17):

| # | Topic | What you'll learn |
|---|-------|-------------------|
| 1 | **Memory** | Agents that remember users & conversations across turns |
| 2 | **Knowledge (RAG)** | Grounding answers in your own documents |
| 3 | **Storage** | Production-grade persistence across restarts |
| 4 | **Workflows** | Deterministic, multi-step agent pipelines |

> 🧵 We continue with **"Safari"**, our travel assistant — now upgrading it from a clever
> chatbot into a stateful concierge.

We use **SQLite** for storage throughout (zero setup, one file on disk) — ideal for
learning. In production you'd swap in Postgres with the *same* API.


---
## 0. Setup

Same setup as notebook 1: load the OpenRouter key and pick a model. (Run notebook 1 first
if you haven't — it explains `uv sync` and the `.env` file.)


In [ ]:
from dotenv import load_dotenv
import os, getpass

load_dotenv()
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

from agno.models.openrouter import OpenRouter

MODEL_ID = "google/gemini-3.1-flash-lite"   # cheap + reliable; swap as you like

def make_model():
    return OpenRouter(id=MODEL_ID)

print(f"✅ Ready. Using model: {MODEL_ID}")


### One database to hold our state

In Agno v2, a single **`Db`** object stores everything an agent needs to persist: session
history, user memories, and knowledge metadata. We'll use a SQLite file.

> 📝 **Heads up:** the slides show the older `agno.storage.sqlite.SqliteStorage` /
> `storage=` API. In current Agno (v2) it's **`agno.db.sqlite.SqliteDb`** passed as
> **`db=`**. This notebook uses the current API.


In [ ]:
from agno.db.sqlite import SqliteDb

# Everything persists into this single file. Delete it to reset all state.
db = SqliteDb(db_file="safari.db")
print("📦 Database ready → safari.db")


---
## 1. Memory: context that persists  *(slide 11)*

Memory is the difference between a **stateless API call** and a **coherent assistant**.
Agno has two complementary kinds of memory:

**A. Session history** — the recent back-and-forth of *this* conversation. Enable it with
`add_history_to_context=True` and control how many past turns to include with
`num_history_runs`.

**B. User memories** — durable facts the agent *learns* about a user (their name,
preferences, constraints) and reuses across *different* sessions. Enable with
`enable_user_memories=True`. These are tied to a **`user_id`** and stored in the `db`.

### 1A. Session history (short-term coherence)

Watch the agent remember something said earlier *in the same session*.


In [ ]:
from agno.agent import Agent

chat = Agent(
    name="Safari",
    model=make_model(),
    db=db,
    add_history_to_context=True,   # 👈 feed recent turns back into the model
    num_history_runs=5,            # how many prior exchanges to include
    instructions="You are Safari, a concise travel assistant.",
    markdown=True,
)

# A session_id groups turns into one conversation.
SESSION = "demo-chat"

chat.print_response("My name is Amara and I'm planning a trip to Tanzania.", session_id=SESSION, stream=True)


In [ ]:
# No re-introduction needed — the agent recalls the earlier turn from session history.
chat.print_response("What's my name, and where am I going again?", session_id=SESSION, stream=True)


### 1B. User memories (long-term personalisation)

Now the powerful part. With `enable_user_memories=True`, the agent **extracts and stores
durable facts** about the user automatically. These persist in the database and are
available in *future* sessions — keyed by `user_id`.


In [ ]:
concierge = Agent(
    name="Safari Concierge",
    model=make_model(),
    db=db,
    enable_user_memories=True,     # 👈 learn & store facts about the user
    instructions="You are Safari, a personal travel concierge. Personalise your advice.",
    markdown=True,
)

USER = "amara@example.com"

# The agent will quietly remember the preferences mentioned here.
concierge.print_response(
    "I'm vegetarian, I love hiking, and my budget is tight. I prefer trains over flights.",
    user_id=USER,
    session_id="trip-1",
    stream=True
)


Let's see what Safari actually **remembered**. The memories live in the database and can
be read back with `get_user_memories`.


In [ ]:
memories = concierge.get_user_memories(user_id=USER)
print(f"🧠 Stored {len(memories)} memories for {USER}:\n")
for m in memories:
    print(" •", m.memory)


Now start a **brand-new session** (`trip-2`) — no history carried over — yet the agent
*still* knows Amara's preferences, because user memories are durable and user-scoped.


In [ ]:
concierge.print_response(
    "Recommend a single destination for my next trip and explain why it fits me.",
    user_id=USER,
    session_id="trip-2",
    stream=True   # fresh conversation, but memories persist
)


---
## 2. Knowledge: grounding answers in your documents  *(slides 12–13)*

**Knowledge** gives an agent access to *your* documents, retrieved at query time — this is
**RAG** (Retrieval-Augmented Generation). It's distinct from memory:

| Memory | Knowledge |
|--------|-----------|
| Conversation history | Documents & data |
| User-specific | Shared across users |
| Built from chat | Loaded from files / DB / URLs |
| "What did *this user* tell me?" | "What do *my docs* say?" |

**How it works:** Ingest documents → split into chunks → embed each chunk into a vector →
store in a vector DB. At query time, the agent embeds the question, finds the most similar
chunks, and answers grounded in them.

### Embeddings without an extra API key
OpenRouter serves chat models but **not embeddings**. So we use **FastEmbed**, which runs a
small embedding model **locally** — free, no extra key. (It downloads a ~70 MB model on
first run.) The vector store is **LanceDB** (an embedded, file-based vector DB).


In [ ]:
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.embedder.fastembed import FastEmbedEmbedder
from agno.vectordb.lancedb import LanceDb, SearchType

vector_db = LanceDb(
    table_name="safari_kb",
    uri="safari_lancedb",                 # stored on disk in this folder
    search_type=SearchType.hybrid,        # vector + keyword search combined
    embedder=FastEmbedEmbedder(),         # local, free embeddings
)

knowledge = Knowledge(
    vector_db=vector_db,
    contents_db=db,                       # tracks what's been ingested
)
print("📚 Knowledge base ready (LanceDB + FastEmbed)")

Let's ingest some **private travel notes** — information the model couldn't possibly know
on its own (a fictional tour operator's policies). `add_content` chunks, embeds, and stores
it. We add a few text snippets; you can also pass `path="..."` for a local file or
`url="..."` for a web page / PDF.


In [ ]:
notes = {
    "kilimanjaro-policy": (
        "Safari Co. Kilimanjaro treks: the Machame route takes 7 days and costs $1,950 "
        "per person, including park fees, meals, and a certified guide. Minimum age is 12. "
        "We provide a free acclimatisation day on day 4."
    ),
    "serengeti-policy": (
        "Safari Co. Serengeti packages: a 4-day group safari is $1,400 per person. "
        "Private vehicle upgrade is +$300. The great wildebeest migration is best viewed "
        "between July and September. All packages are vegetarian-friendly on request."
    ),
    "booking-policy": (
        "Safari Co. bookings require a 20% deposit. Free cancellation up to 30 days before "
        "departure; 50% refund within 30 days. We do not offer flights, only ground tours."
    ),
}

for name, text in notes.items():
    knowledge.add_content(name=name, text_content=text)

print("✅ Ingested", len(notes), "documents into the knowledge base")


Now attach the knowledge base to an agent with `search_knowledge=True`. The agent will
**search the docs automatically** before answering. Ask it something only our private notes
contain:


In [ ]:
kb_agent = Agent(
    name="Safari Co. Advisor",
    model=make_model(),
    knowledge=knowledge,
    search_knowledge=True,    # 👈 retrieve relevant chunks before answering
    instructions=[
        "You are an advisor for Safari Co.",
        "Answer using ONLY the company's knowledge base. If it's not there, say so.",
        "Quote specific prices and policies when relevant.",
    ],
    markdown=True,
)

kb_agent.print_response("How much is the Kilimanjaro trek, how long is it, and what's the minimum age?", stream=True)


In [ ]:
# A question that spans two documents — and respects the cancellation policy.
kb_agent.print_response("I'm vegetarian. Can I do the Serengeti safari, and what happens if I cancel 2 weeks before?",stream=True)


---
## 3. Storage: production-grade persistence  *(slides 14–15)*

Production agents run across **sessions, users, and restarts**. Storage persists agent
state to a database so nothing is lost. We've *already* been using it — every `db=`
attached above wrote sessions and memories to `safari.db`.

> **Why storage matters:** Session recovery (resume after a crash/restart) · Multi-user
> scaling (isolate state per `user_id`) · Audit & debug (a full trace of what happened).

### 3A. Resuming a session after a "restart"

We'll simulate a process restart by creating a **brand-new agent object** that shares the
**same `db` and `session_id`**. It picks up the conversation exactly where it left off —
no in-memory state required.


In [ ]:
# Earlier (in section 1A) the "demo-chat" session learned that Amara is going to Tanzania.
# Imagine the app restarted — we build a fresh agent pointed at the same db + session.
resumed = Agent(
    name="Safari",
    model=make_model(),
    db=db,                          # same database file
    add_history_to_context=True,
    num_history_runs=5,
    instructions="You are Safari, a concise travel assistant.",
)

resumed.print_response(
    "Based on our earlier chat, suggest one thing I should book first.",
    session_id="demo-chat",         # same session id → history is restored
    stream=True
)


### 3B. Audit & debug: read state straight from the database

Because everything is persisted, you can inspect it independently of any agent — useful for
debugging, analytics, or building an admin dashboard.


In [ ]:
from agno.db.base import SessionType

sessions = db.get_sessions(session_type=SessionType.AGENT)
print(f"🗂️  {len(sessions)} agent session(s) stored in safari.db:")
for s in sessions:
    print("   •", s.session_id)

print("\n🧠 User memories on record:")
for m in db.get_user_memories(user_id="amara@example.com"):
    print("   •", m.memory)


> 🔁 **Same code, bigger database.** To go to production you'd swap `SqliteDb` for
> `PostgresDb(db_url=...)` — the agent code above stays *identical*. That's the payoff of
> Agno's storage abstraction.


---
## 4. Workflows: multi-step orchestration  *(slides 16–17)*

Teams (notebook 1) let a leader **dynamically** decide who does what. **Workflows** are the
opposite: **deterministic, explicit pipelines** where *you* define the steps and their
order. You get orchestration, repeatability, and full traceability — ideal when the process
is known in advance.

We'll build a **3-step trip-planning pipeline**, reusing the components we've built:

```
   Research  ──▶  Plan  ──▶  Review
  (web facts)   (itinerary)  (polish & sanity-check)
```

Each `Step` wraps an agent; the `Workflow` runs them in sequence, passing each step's
output to the next.


In [ ]:
from agno.tools.duckduckgo import DuckDuckGoTools

research_agent = Agent(
    name="Researcher",
    model=make_model(),
    tools=[DuckDuckGoTools()],
    instructions="Gather key facts about the destination: top sights, best season, getting around.",
)

planner_agent = Agent(
    name="Planner",
    model=make_model(),
    instructions="Turn the research into a concrete day-by-day itinerary. Be specific and realistic.",
)

reviewer_agent = Agent(
    name="Reviewer",
    model=make_model(),
    instructions="Review the itinerary for pacing and feasibility. Fix issues and produce the final polished plan.",
)


In [ ]:
from agno.workflow import Workflow, Step

trip_workflow = Workflow(
    name="Trip Planning Pipeline",
    db=db,                                  # workflows persist their runs too
    steps=[
        Step(name="Research", agent=research_agent),
        Step(name="Plan", agent=planner_agent),
        Step(name="Review", agent=reviewer_agent),
    ],
)
print("🛠️  Workflow assembled:", [s.name for s in trip_workflow.steps])


In [ ]:
trip_workflow.print_response(
    "Plan a relaxed 3-day cultural trip to Marrakech, Morocco for a first-time visitor.",
    stream=True,
)


You just watched **input flow through Research → Plan → Review** in a structured,
traceable pipeline. Each step is a focused agent; the workflow guarantees the order and
hands data along.

**Workflow benefits:** Orchestration (chain agents & tools) · Error handling (retries &
fallbacks per step) · Observability (trace every step and decision).

---
## 🎉 Recap — Advanced Session

You upgraded Safari from a stateless chatbot into a stateful, production-shaped system:

1. **Memory** — session history for in-conversation coherence, plus durable **user
   memories** (`enable_user_memories`, `user_id`) that persist across sessions.
2. **Knowledge (RAG)** — grounded answers from *your* documents using LanceDB + local
   FastEmbed embeddings, retrieved automatically with `search_knowledge=True`.
3. **Storage** — a single `SqliteDb` persists sessions, memories and knowledge; agents
   **resume after restarts** and you can **audit state** straight from the DB. Swap to
   Postgres with no code changes.
4. **Workflows** — deterministic, multi-step pipelines (`Step` + `Workflow`) for
   repeatable, traceable orchestration.

### The full picture
**Model + Instructions + Tools** → an agent that acts · **Structured outputs** → reliable
data · **Teams** → collaboration · **Memory + Knowledge + Storage** → state · **Workflows**
→ orchestration. That's the complete Agno toolkit for building *useful* AI agents. 🚀

> 🧹 To reset everything and re-run from scratch, delete `safari.db` and the
> `safari_lancedb/` folder.
